# spinangle — gated spherical nGPT-JEPA vs. official LeWM (Colab GPU)

Official install: `uv` + isolated **Python 3.10** venv + `stable-worldmodel[train]` plus the env deps the chosen benchmark uses (the full `[env]` extra bundles Crafter/Atari, which aren't LeWM benchmarks and break resolution). Everything runs through the venv interpreter; eval renders headless via `xvfb`+EGL.

**Setup:** Runtime → GPU. Start with `BENCH='tworoom'` + `EPOCHS=5`. Every step is loud and stops at the real error.

## ▶️ The big cell (edit config, run)

In [ ]:
#@title 🌀 spinangle: gated spherical nGPT-JEPA vs official LeWM — REAL install + run
import os, subprocess, sys, glob

# ----------------------------- config -----------------------------
BENCH    = "tworoom"   # tworoom (3.4G) | pusht (13G) | reacher (24G) | cube (46G — big disk)
EPOCHS   = 5           # 5 = validate; 100 = matched-compute comparison
VARIANTS = ["official_lewm", "gated_spherical"]
GET_DATA = True
BRANCH   = "claude/upbeat-babbage-kbmgsr"
GH_TOKEN = ""          # only if repo private
# ------------------------------------------------------------------

DATACFG = {"tworoom": "tworoom", "pusht": "pusht", "reacher": "dmc", "cube": "ogb"}[BENCH]
# env deps the benchmark ACTUALLY uses (the full [env] extra also pulls Crafter+Atari,
# which are not LeWM benchmarks and break dependency resolution):
ENV_DEPS = {"tworoom": "pygame pymunk shapely", "pusht": "pygame pymunk shapely",
            "reacher": "dm_control mujoco", "cube": "ogbench"}[BENCH]
H = "/content/stable-wm"; VENV = "/content/lewmenv"; PY = f"{VENV}/bin/python"
os.environ.update(STABLEWM_HOME=H, MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl")

def run(c, check=True):
    print(f"\n\033[1;36m$ {c}\033[0m", flush=True)
    return subprocess.run(c, shell=True, check=check).returncode

def cap(c):  # capture + print (so resolver errors are visible)
    print(f"\n\033[1;36m$ {c}\033[0m", flush=True)
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print(((r.stdout or "") + (r.stderr or ""))[-6000:]); print("exit", r.returncode)
    return r.returncode

try:
    from google.colab import userdata
    GH_TOKEN = GH_TOKEN or (userdata.get("GITHUB_TOKEN") or "")
except Exception:
    pass

run("nvidia-smi -L || echo '⚠️  NO GPU — Runtime > Change runtime type > GPU'", check=False)

# 0) clone (public; token only if private) -------------------------------------
if not os.path.isdir("/content/spinangle/.git"):
    auth = f"{GH_TOKEN}@" if GH_TOKEN else ""
    run(f"git clone -b {BRANCH} https://{auth}github.com/turtlenottortoise/spinangle.git /content/spinangle")
else:
    run("cd /content/spinangle && git pull", check=False)
os.chdir("/content/spinangle")

# 1) headless render libs for eval (pygame + MuJoCo) ---------------------------
run("apt-get -qq update && apt-get -qq install -y xvfb zstd ffmpeg patchelf "
    "libegl1 libgl1-mesa-glx libosmesa6 libglfw3 libglew2.2 >/dev/null 2>&1", check=False)

# 2) uv + isolated Python 3.10 venv (reuse if present) -------------------------
if not os.path.exists(PY):
    run("pip install -q uv")
    run("uv python install 3.10")
    run(f"uv venv --python 3.10 {VENV}")

# 3) install: [train] is REQUIRED; then env deps (full [env] first, scoped fallback)
if cap(f"uv pip install --python {PY} 'stable-worldmodel[train]' matplotlib huggingface_hub"):
    raise SystemExit("❌ [train] stack failed to install — paste the output above.")
if cap(f"uv pip install --python {PY} 'stable-worldmodel[env]'"):
    print("\n[note] full [env] extra did not resolve (it bundles Crafter/craftax + "
          "Atari/ale-py, which are NOT LeWM benchmarks). Installing this benchmark's "
          f"env deps instead: {ENV_DEPS}")
    if cap(f"uv pip install --python {PY} {ENV_DEPS}"):
        raise SystemExit(f"❌ env deps for {BENCH} failed — paste the output above.")

# 4) ensure CUDA torch in the venv --------------------------------------------
if subprocess.run(f"{PY} -c \"import torch,sys;sys.exit(0 if torch.cuda.is_available() else 1)\"",
                  shell=True).returncode:
    run(f"uv pip install --python {PY} torch torchvision --index-url https://download.pytorch.org/whl/cu124")
run(f"{PY} -c \"import torch,hydra,stable_worldmodel,stable_pretraining; "
    f"print('torch',torch.__version__,'cuda',torch.cuda.is_available(),'| stack OK')\"")

# 5) harness sanity ------------------------------------------------------------
run(f"{PY} smoke_test.py && {PY} metrics.py")

# 6) data + checkpoint; PHASE 1 reproduce official LeWM (eval renders -> xvfb) --
run(f"{PY} scripts/download_assets.py --benchmark {BENCH} --ckpt" + (" --data" if GET_DATA else ""))
run(f"xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/lewm")

# 7) train + eval variants -----------------------------------------------------
CKPT = f"{H}/checkpoints/{BENCH}"
for v in VARIANTS:
    run(f"{PY} train.py +experiment={v} data={DATACFG} "
        f"output_model_name={BENCH}/{v} trainer.max_epochs={EPOCHS} wandb.enabled=false")
    for old in sorted(glob.glob(f"{CKPT}/{v}/weights_epoch_*.pt"), key=os.path.getmtime)[:-1]:
        os.remove(old)
    run(f"xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/{v}")
    sph = "" if v in ("official_lewm", "lewm_nosigreg") else "--spherical"
    run(f"{PY} scripts/eval_latent_metrics.py --policy {BENCH}/{v} --data {DATACFG} "
        f"--benchmark {BENCH} --variant {v} {sph} --horizon 20 --num_batches 16", check=False)

# 8) plots ---------------------------------------------------------------------
run(f"{PY} scripts/make_plots.py")
from IPython.display import Image, display
for p in ["success_vs_steps", "rollout_error_vs_horizon", "retrieval_vs_steps",
          "rank_clumping", "planning_budget_curve"]:
    fp = f"/content/spinangle/plots/{p}.png"
    if os.path.exists(fp):
        display(Image(fp))
print("\n✅ DONE — results in results/all_runs.csv, plots in plots/.")


## Phase 7 — νGPT scaling (optional; run after the loop above)

In [ ]:
PY, BENCH, DATACFG, EPOCHS = '/content/lewmenv/bin/python', 'tworoom', 'tworoom', 100
for v in ['gated_spherical', 'ngpt_lr', 'ngpt_lr_groups']:
    !{PY} train.py +experiment={v} data={DATACFG} output_model_name={BENCH}/{v} \
        trainer.max_epochs={EPOCHS} wandb.enabled=false
    !xvfb-run -a {PY} eval.py --config-name={BENCH}.yaml policy={BENCH}/{v}


## Persist results back to the branch (optional)

In [ ]:
!cd /content/spinangle && git add results/all_runs.csv plots/*.png && \
  git -c user.email=colab@local -c user.name=colab commit -m 'colab: results' && \
  git push || echo 'configure git auth (token) to push'
